# copilot

> GitHub Copilot models through the `Chat` API, using a Copilot subscription instead of a vendor key.

Copilot uses an OpenAI-compatible chat endpoint with its own authentication. Each request needs a
short-lived token and editor headers. This module adds that authentication to `RemoteChat`.

This integration is reverse-engineered and unsupported by GitHub. The Copilot subscription terms
apply.


In [ ]:
#| default_exp copilot

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, os, time
import httpx
from dataclasses import dataclass, replace
from fastcore.all import L, Path, store_attr, first
from rishi.core import *
from rishi.remote import RemoteChat

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec', 'ToolCall']

In [ ]:
from fastcore.test import test_eq, test_fail

## The wire

Copilot serves OpenAI chat completions at `{endpoint}/chat/completions`. The token exchange returns
the endpoint for the account's plan. `COPILOT_API` is the fallback when the response omits it.

Copilot requires headers from a recognised editor. `copilot-integration-id` identifies the editor.
The version headers match the VS Code chat extension. `x-initiator` distinguishes user turns from
tool-loop turns. `copilot-vision-request` enables image input.

Editors also send `x-request-id`, but Rishi omits it. fastllm caches HTTP clients by constructor
arguments. A changing request id would create a new client for every request.

In [ ]:
#| export
#: Where Copilot listens when the token exchange names no endpoint of its own.
COPILOT_API = 'https://api.githubcopilot.com'
#: Trades a GitHub OAuth token for a short-lived Copilot one.
COPILOT_TOKEN_URL = 'https://api.github.com/copilot_internal/v2/token'
#: The OAuth app the editors sign in as, and the only client id GitHub mints Copilot tokens for.
COPILOT_CLIENT_ID = 'Iv1.b507a08c87ecfe98'

#: Which editor Copilot is told it is talking to. Rebind these to say otherwise.
EDITOR_VERSION = 'vscode/1.99.3'
PLUGIN_VERSION = 'copilot-chat/0.26.7'
USER_AGENT     = 'GitHubCopilotChat/0.26.7'
INTEGRATION_ID = 'vscode-chat'
API_VERSION    = '2025-04-01'

def copilot_hdrs(auth=None, *, integration_id=INTEGRATION_ID, vision=False, initiator=None,
                 intent='conversation-panel'):
    "Headers Copilot wants on every request. `auth` is a whole `Authorization` value."
    h = {'editor-version': EDITOR_VERSION, 'editor-plugin-version': PLUGIN_VERSION,
         'user-agent': USER_AGENT, 'copilot-integration-id': integration_id,
         'x-github-api-version': API_VERSION}
    if intent: h['openai-intent'] = intent
    if auth: h['Authorization'] = auth
    if vision: h['copilot-vision-request'] = 'true'
    if initiator: h['x-initiator'] = initiator
    return h

In [ ]:
h = copilot_hdrs()
test_eq(h['copilot-integration-id'], 'vscode-chat')
assert 'Authorization' not in h                      # fastllm sets it from the api key
assert 'x-request-id' not in h                       # would defeat fastllm's client cache
test_eq(copilot_hdrs('Bearer t')['Authorization'], 'Bearer t')
test_eq(copilot_hdrs(vision=True)['copilot-vision-request'], 'true')
test_eq(copilot_hdrs(initiator='agent')['x-initiator'], 'agent')

## The GitHub token

Copilot requires a GitHub OAuth token issued to a recognised application. Personal access tokens do
not work. Editor sign-ins store a usable token in a configuration file. `copilot_login()` stores the
same token shape under `~/.config/rishi`.

`copilot_oauth()` checks Copilot-specific environment variables first, then editor files. It checks
`GH_TOKEN` and `GITHUB_TOKEN` last because they often contain personal access tokens. The file
reader finds a nested `oauth_token` key.

In [ ]:
#| export
COPILOT_ENVS = ('GITHUB_COPILOT_OAUTH_TOKEN', 'GH_COPILOT_TOKEN')
GITHUB_ENVS = ('GH_TOKEN', 'GITHUB_TOKEN')
OAUTH_ENVS = COPILOT_ENVS + GITHUB_ENVS   #: every variable read, though not all at the same moment

def oauth_paths():
    "Files a Copilot sign-in may have left a GitHub OAuth token in, best first."
    cfg = Path(os.getenv('XDG_CONFIG_HOME') or Path.home()/'.config')
    ps  = [cfg/'rishi'/'copilot.json'] + [cfg/'github-copilot'/f for f in ('apps.json', 'hosts.json')]
    if (la := os.getenv('LOCALAPPDATA')):
        ps += [Path(la)/'github-copilot'/f for f in ('apps.json', 'hosts.json')]
    return ps + [Path.home()/'.copilot'/'config.json']

def find_oauth(o, depth=0):
    "The `oauth_token` in a decoded config file, however deep the editor nested it."
    if depth > 4 or not isinstance(o, dict): return None
    if isinstance(t := o.get('oauth_token'), str) and t: return t
    return first(x for x in (find_oauth(v, depth+1) for v in o.values()) if x)

def read_oauth(p):
    "The GitHub OAuth token in config file `p`, or `None` if it has none or is not there."
    try: return find_oauth(json.loads(Path(p).read_text()))
    except Exception: return None

def _env_oauth(ks):
    "The first of environment variables `ks` that holds anything."
    return first(t for k in ks if (t := os.getenv(k)))

def copilot_oauth(token=None):
    "A GitHub OAuth token: what you passed, else `COPILOT_ENVS`, else an editor sign-in, else `GITHUB_ENVS`."
    if token: return token
    if (t := _env_oauth(COPILOT_ENVS)): return t
    for p in oauth_paths():
        if (t := read_oauth(p)): return t
    if (t := _env_oauth(GITHUB_ENVS)): return t
    raise RuntimeError(
        'No GitHub OAuth token for Copilot. Set one of ' + ', '.join(OAUTH_ENVS) + ', or sign in to '
        'Copilot in an editor, or run `from rishi.copilot import copilot_login; copilot_login()`.')

In [ ]:
# every shape an editor writes, and one that holds nothing
test_eq(find_oauth({'oauth_token': 'a'}), 'a')
test_eq(find_oauth({'github.com:Iv1.b507a08c87ecfe98': {'oauth_token': 'b', 'user': 'me'}}), 'b')
test_eq(find_oauth({'hosts': {'github.com': {'oauth_token': 'c'}}}), 'c')
test_eq(find_oauth({'user': 'me'}), None)
test_eq(read_oauth('/nonexistent/apps.json'), None)

test_eq(copilot_oauth('explicit'), 'explicit')
test_eq(oauth_paths()[0].name, 'copilot.json')          # rishi's own file is read before any editor's
assert any('github-copilot' in str(p) for p in oauth_paths())

In [ ]:
# an editor sign-in beats a general-purpose GitHub variable, because that one holds a personal
# access token, which Copilot refuses. Only a Copilot-specific variable beats the sign-in.
import tempfile

with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'apps.json'
    f.write_text(json.dumps({'github.com:Iv1.b507a08c87ecfe98': {'oauth_token': 'gho_editor'}}))
    saved = {k: os.environ.pop(k, None) for k in OAUTH_ENVS}
    orig_paths, oauth_paths = oauth_paths, lambda: [f]
    try:
        os.environ['GITHUB_TOKEN'] = 'ghp_pat'
        test_eq(copilot_oauth(), 'gho_editor')          # the sign-in, not the PAT
        os.environ['GH_COPILOT_TOKEN'] = 'gho_said_so'
        test_eq(copilot_oauth(), 'gho_said_so')         # names Copilot outright, so it wins
        del os.environ['GH_COPILOT_TOKEN']
        oauth_paths = lambda: []
        test_eq(copilot_oauth(), 'ghp_pat')             # with no sign-in to hide, it is all there is
        del os.environ['GITHUB_TOKEN']
        test_fail(copilot_oauth, contains='No GitHub OAuth token')
    finally:
        oauth_paths = orig_paths
        for k, v in saved.items():
            os.environ.pop(k, None)
            if v is not None: os.environ[k] = v

## The Copilot token

The GitHub OAuth token is exchanged for a Copilot token that lasts about 30 minutes. `CopilotAuth`
renews it before expiry. Chats that share one `CopilotAuth` also share the exchange.

`api_key=` accepts an existing Copilot token. Rishi cannot renew it without the GitHub OAuth token.

In [ ]:
#| export
@dataclass(frozen=True)
class CopilotToken:
    "One minted Copilot token: the bearer, the endpoint it is good for, and when it dies."
    key: str
    base_url: str = COPILOT_API
    expires_at: float = 0.      # unix seconds; 0 for a token rishi did not mint and cannot time

    def stale(self, skew=300):
        "Is it inside `skew` seconds of expiry? An untimed token is never stale."
        return bool(self.expires_at) and time.time() >= self.expires_at - skew

def copilot_exchange(oauth=None, *, url=COPILOT_TOKEN_URL, timeout=30):
    "Trade a GitHub OAuth token for a `CopilotToken`."
    r = httpx.get(url, timeout=timeout,
                  headers={**copilot_hdrs(f'token {copilot_oauth(oauth)}'), 'Accept': 'application/json'})
    if r.status_code in (401, 403, 404): raise PermissionError(
        f'GitHub refused the Copilot token exchange ({r.status_code}). That account needs an active '
        'Copilot subscription, and the OAuth token has to come from an app Copilot knows: an editor '
        'sign-in or `copilot_login()`, not a personal access token. A personal access token is what '
        'the 404 means: to an app Copilot does not know, the endpoint is not there at all.')
    r.raise_for_status()
    d = r.json()
    exp = float(d.get('expires_at') or 0)
    if exp > 1e11: exp /= 1000      # some responses count in milliseconds
    return CopilotToken(d['token'], (d.get('endpoints') or {}).get('api') or COPILOT_API, exp)

class CopilotAuth:
    "A Copilot token that re-mints itself. Share one and every chat holding it stays signed in."
    def __init__(self, oauth=None, *, key=None, base_url=None, skew=300):
        store_attr()
        self.tok = CopilotToken(key, base_url or COPILOT_API) if key else None

    def token(self, force=False):
        "The live token, minted when there is none and re-minted when it is nearly out of time."
        if self.key: return self.tok    # yours, and rishi has nothing to renew it with
        if force or self.tok is None or self.tok.stale(self.skew):
            t = copilot_exchange(self.oauth)
            self.tok = replace(t, base_url=self.base_url) if self.base_url else t
        return self.tok

    def __repr__(self):
        t = self.tok
        return f'CopilotAuth({"signed in" if t else "not yet"}, {t.base_url if t else COPILOT_API})'

def copilot_catalog(auth=None, timeout=30):
    "Every model this account can reach, as `{id: entry}` the way Copilot describes them."
    t = (auth or CopilotAuth()).token()
    r = httpx.get(f"{t.base_url.rstrip('/')}/models", headers=copilot_hdrs(f'Bearer {t.key}'), timeout=timeout)
    r.raise_for_status()
    return {m['id']: m for m in (r.json().get('data') or []) if m.get('id')}

def copilot_models(auth=None, timeout=30, kind='chat'):
    "Model ids this account can reach. Copilot is the authority, not a table in here. `kind=None` keeps the embedding and completion ones too."
    return [i for i, m in copilot_catalog(auth, timeout).items()
            if not kind or (m.get('capabilities') or {}).get('type') == kind]

def copilot_ctx(entry):
    "The prompt window Copilot reports for one catalogue entry, or `None` where it reports none."
    lim = ((entry or {}).get('capabilities') or {}).get('limits') or {}
    return lim.get('max_prompt_tokens') or lim.get('max_context_window_tokens') or None

In [ ]:
# the catalogue carries more than ids: what each model is for, and how big its window is. An
# embedding model answering a chat request is a 400 the caller cannot read, so `kind` filters.
_payload = {'data': [
    {'id': 'gpt-5.5', 'capabilities': {'type': 'chat', 'limits': {'max_prompt_tokens': 128000}}},
    {'id': 'claude-opus-4.7', 'capabilities': {'type': 'chat', 'limits': {'max_context_window_tokens': 264000}}},
    {'id': 'gpt-41-copilot', 'capabilities': {'type': 'completion'}},
    {'id': 'text-embedding-3-small', 'capabilities': {'type': 'embeddings'}},
    {'no_id': 'skipped'}]}

class _ModelsResp:
    status_code = 200
    def raise_for_status(self): pass
    def json(self): return _payload

orig, httpx.get = httpx.get, lambda *a, **kw: _ModelsResp()
try:
    auth = CopilotAuth(key='tid=abc')                 # a key of its own, so nothing is exchanged
    cat = copilot_catalog(auth)
    test_eq(list(cat), ['gpt-5.5', 'claude-opus-4.7', 'gpt-41-copilot', 'text-embedding-3-small'])
    test_eq(copilot_models(auth), ['gpt-5.5', 'claude-opus-4.7'])          # chat only, by default
    test_eq(len(copilot_models(auth, kind=None)), 4)                       # everything it named
    test_eq(copilot_models(auth, kind='embeddings'), ['text-embedding-3-small'])
    test_eq(copilot_ctx(cat['gpt-5.5']), 128000)                           # max_prompt_tokens
    test_eq(copilot_ctx(cat['claude-opus-4.7']), 264000)                   # else the whole window
    test_eq(copilot_ctx(cat['gpt-41-copilot']), None)                      # it reports no limits
    test_eq(copilot_ctx(None), None)
finally: httpx.get = orig

In [ ]:
now = time.time()
test_eq(CopilotToken('k').stale(), False)                     # untimed: never stale
test_eq(CopilotToken('k', expires_at=now+3600).stale(), False)
test_eq(CopilotToken('k', expires_at=now+60).stale(), True)    # inside the 5 minute skew
test_eq(CopilotToken('k', expires_at=now-1).stale(), True)

# a key you brought yourself is used as-is, and never exchanged
test_eq(CopilotAuth(key='tid=abc').token().key, 'tid=abc')
test_eq(CopilotAuth(key='k').token().base_url, COPILOT_API)
test_eq(CopilotAuth(key='k', base_url='https://api.business.githubcopilot.com').token().base_url,
        'https://api.business.githubcopilot.com')

# minting runs once, then not again until the token is nearly out of time
mints = []
def _fake_exchange(oauth=None, **kw):
    mints.append(oauth)
    return CopilotToken('tid=minted', 'https://api.individual.githubcopilot.com', time.time()+1800)

orig, copilot_exchange = copilot_exchange, _fake_exchange
try:
    a = CopilotAuth('gho_x')
    test_eq(a.token().key, 'tid=minted')
    test_eq(a.token().base_url, 'https://api.individual.githubcopilot.com')
    test_eq(len(mints), 1)                                     # the second call reused it
    a.tok = replace(a.tok, expires_at=time.time()+10)
    a.token()
    test_eq(len(mints), 2)                                     # nearly expired, so re-minted
finally: copilot_exchange = orig

In [ ]:
# a personal access token gets a 404 out of the exchange, and that is a refusal like any other:
# what the caller needs told is that Copilot takes no PAT, not that some URL was not found
class _FakeResp:
    def __init__(self, code): self.status_code = code
    def raise_for_status(self): raise AssertionError('a refusal must not reach raise_for_status')

def _refused(code):
    orig, httpx.get = httpx.get, lambda *a, **kw: _FakeResp(code)
    try: copilot_exchange('gho_x')
    except PermissionError as e: return str(e)
    finally: httpx.get = orig
    assert False, f'{code} raised no PermissionError'

for code in (401, 403, 404): assert str(code) in _refused(code)

## Signing in

`copilot_login()` runs GitHub's device flow with the editor client id. It prints a code and waits for
browser authorization. It saves the token where `copilot_oauth()` looks first. An existing editor
sign-in makes this step unnecessary.

In [ ]:
#| export
DEVICE_CODE_URL  = 'https://github.com/login/device/code'   #: GitHub's device flow
ACCESS_TOKEN_URL = 'https://github.com/login/oauth/access_token'

def save_oauth(token, path=None):
    "Write a GitHub OAuth token where `copilot_oauth` looks first, readable only by you."
    p = Path(path or oauth_paths()[0])
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({'oauth_token': token}))
    p.chmod(0o600)
    return p

def copilot_login(client_id=COPILOT_CLIENT_ID, *, scope='read:user', save=True, printer=print, timeout=30):
    "Run GitHub's device flow: print a code, wait while you enter it, return the OAuth token."
    hdrs = {'Accept': 'application/json', 'User-Agent': USER_AGENT}
    d = httpx.post(DEVICE_CODE_URL, json={'client_id': client_id, 'scope': scope},
                   headers=hdrs, timeout=timeout).json()
    if not d.get('device_code'): raise RuntimeError(f'GitHub declined the device code request: {d}')
    printer(f"Open {d['verification_uri']} and enter {d['user_code']}")
    wait, deadline = float(d.get('interval') or 5), time.time() + float(d.get('expires_in') or 900)
    body = {'client_id': client_id, 'device_code': d['device_code'],
            'grant_type': 'urn:ietf:params:oauth:grant-type:device_code'}
    while time.time() < deadline:
        time.sleep(wait)
        r = httpx.post(ACCESS_TOKEN_URL, json=body, headers=hdrs, timeout=timeout).json()
        if (tok := r.get('access_token')):
            if save: printer(f'Saved to {save_oauth(tok)}')
            return tok
        if r.get('error') == 'slow_down': wait += float(r.get('interval') or 5)   # GitHub asks for a longer poll
        elif r.get('error') != 'authorization_pending':
            raise RuntimeError(f"device flow failed: {r.get('error_description') or r.get('error')}")
    raise TimeoutError('the device code expired before it was entered')

## CopilotChat

`CopilotChat._kw` adds the endpoint, live token and required headers to `RemoteChat`. Tools,
streaming, approval and the tool loop keep the shared implementation. Use
`Chat('copilot/gpt-4.1')` or `runtime='copilot'`.

Automatic routing requires the `copilot/` prefix. Bare `gpt-...` and `claude-...` ids continue to
use hosted APIs through `remote`.

In [ ]:
#| export
#: What `CopilotChat()` asks for when you name no model. `copilot_models()` is the authority on what
#: your account can reach, and that list moves faster than a constant in here could.
copilot_default = 'gpt-4.1'

class CopilotChat(RemoteChat):
    "Chat against a GitHub Copilot model: `RemoteChat` pointed at Copilot, with its headers and its half-hour token."
    _runtime = 'copilot'
    local = False   #: the subscription is yours, the machine is GitHub's

    def __init__(self, model=None, *, oauth_token=None, api_key=None, base_url=None, auth=None,
                 integration_id=INTEGRATION_ID, hdrs=None, **kw):
        self.auth = auth or CopilotAuth(oauth_token, key=api_key, base_url=base_url)
        self.integration_id, self.hdrs = integration_id, dict(hdrs or {})
        super().__init__(model or copilot_default, **kw)

    def _has_media(self):
        "Does the history carry an image or audio part? Copilot gates those behind a header."
        return any(isinstance(p, dict) and p.get('type') in ('image_url', 'input_audio')
                   for m in self.hist for p in L(m.get('content')))

    def _initiator(self):
        "`user` for a turn a person opened, `agent` once the model is round-tripping its own tool calls."
        return 'agent' if any(m.get('role') in ('assistant', 'tool') for m in self.hist) else 'user'

    def _kw(self, stream=False, max_output_tokens=None):
        "`RemoteChat._kw` plus Copilot's endpoint, a live token, and the headers it insists on."
        kw = super()._kw(stream, max_output_tokens)
        t  = self.auth.token()
        kw.update(api_key=t.key, base_url=t.base_url, vendor_name=None, api_name='openai_chat')
        kw['xtra_hdrs'] = {**copilot_hdrs(integration_id=self.integration_id, vision=self._has_media(),
                                          initiator=self._initiator()),
                           **(kw.get('xtra_hdrs') or {}), **self.hdrs}
        return kw

In [ ]:
# nothing here reaches the network: the chat is handed a token, and what is asserted is the request
c = CopilotChat('copilot/gpt-4.1', api_key='tid=abc', sp='Be terse.', messages=['hello'])
test_eq(c.runtime, 'copilot')
test_eq(c.model_id, 'gpt-4.1')          # the `copilot/` prefix is stripped, the id is not
test_eq(c.local, False)

kw = c._kw()
test_eq(kw['api_key'], 'tid=abc')
test_eq(kw['base_url'], COPILOT_API)
test_eq(kw['api_name'], 'openai_chat')  # fastllm has no Copilot vendor, so the api is named outright
test_eq(kw['vendor_name'], None)
test_eq(kw['system'], 'Be terse.')
test_eq(kw['xtra_hdrs']['copilot-integration-id'], 'vscode-chat')
test_eq(kw['xtra_hdrs']['x-initiator'], 'user')             # only a user message so far
assert 'copilot-vision-request' not in kw['xtra_hdrs']
assert 'Authorization' not in kw['xtra_hdrs']               # fastllm sets it, and once

# a tool loop is an agent to Copilot, and an image needs the vision header
c.hist += [{'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('ls', {})]},
           {'role': 'tool', 'tool_call_id': 'x', 'name': 'ls', 'content': 'a.py'}]
test_eq(c._kw()['xtra_hdrs']['x-initiator'], 'agent')
c.hist.append({'role': 'user', 'content': [{'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,aa'}}]})
test_eq(c._kw()['xtra_hdrs']['copilot-vision-request'], 'true')

# your headers win over rishi's, so an integration id or an editor version can be swapped out
test_eq(CopilotChat(api_key='k', integration_id='copilot-cli', hdrs={'editor-version': 'vim/9.1'}
                    )._kw()['xtra_hdrs']['editor-version'], 'vim/9.1')
test_eq(CopilotChat(api_key='k', integration_id='copilot-cli')._kw()['xtra_hdrs']['copilot-integration-id'],
        'copilot-cli')

# routing, without building anything
test_eq(resolve_runtime('copilot/claude-sonnet-4.5'), ('copilot', 'claude-sonnet-4.5'))
test_eq(resolve_runtime('claude-sonnet-4.5'), ('remote', 'claude-sonnet-4.5'))   # bare ids stay remote
test_eq(get_runtime('copilot').__name__, 'CopilotChat')   # the exported class, not this cell's

In [ ]:
# and what fastllm builds from those arguments is the request Copilot actually wants
from fastllm.acomplete import mk_client

kw = CopilotChat('copilot/gpt-4.1', api_key='tid=abc', sp='Be terse.', messages=['hi'])._kw()
cli, api_name, vendor = mk_client(model='gpt-4.1', vendor_name=kw['vendor_name'], api_name=kw['api_name'],
                                  api_key=kw['api_key'], base_url=kw['base_url'], xtra_hdrs=kw['xtra_hdrs'])
op = cli.chat.create_chat_completion
test_eq(op.base_url + op.path, 'https://api.githubcopilot.com/chat/completions')

hdrs = dict(cli.transport.base_headers)
test_eq([k for k in hdrs if k.lower() == 'authorization'], ['Authorization'])   # one, not two
test_eq(hdrs['Authorization'], 'Bearer tid=abc')
test_eq(hdrs['copilot-integration-id'], 'vscode-chat')
test_eq(hdrs['editor-version'], EDITOR_VERSION)


## Against the real endpoint

These examples require a Copilot subscription and use `eval: false`.

In [ ]:
#| eval: false
from rishi.copilot import copilot_models, copilot_login

# copilot_login()          # only on a machine with no editor signed in
print(copilot_models()[:10])

In [ ]:
#| eval: false
chat = Chat('copilot/gpt-4.1', sp='You are concise.')
r = chat('Reply with exactly: pong')
assert 'pong' in resp_text(r).lower()
print(chat.use)

In [ ]:
#| eval: false
display_stream(Chat('copilot/claude-sonnet-4.5')('Write two sentences about the monsoon.', stream=True))

In [ ]:
#| eval: false
def word_count(text: str) -> int:
    "Count the words in `text`."
    return len(text.split())

chat = Chat('copilot/gpt-4.1', tools=[word_count])   # no approve=, so every call runs
print(resp_text(chat('How many words are in "the quick brown fox"? Use the tool.')))

In [ ]:
#| eval: false
# one exchange, several chats: hand them all the same auth
from rishi.copilot import CopilotAuth, CopilotChat

auth = CopilotAuth()
a, b = CopilotChat('gpt-4.1', auth=auth), CopilotChat('claude-sonnet-4.5', auth=auth)
print(auth)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()